# 02 - בניית הגרף

נוטבוק זה הופך את פיד ה-GTFS הישראלי הגולמי ל**אובייקט הגרף שכל שלב מאוחר יותר בפרויקט מנתח**. המודל הוא *גרף שכנויות-נסיעה* (trip-adjacency graph): צומת הוא תחנה שמשורתת בפועל על ידי לפחות נסיעה אחת, וקשת מכוונת `u -> v` קיימת כאשר קיימת נסיעה כלשהי העוצרת ב-`u` ומיד לאחר מכן עוצרת ב-`v`. משקל הקשת הוא מספר הנסיעות החוצות את אותו מקטע, כלומר תדירות השירות על אותו קטע מסילה/כביש. עיקר העבודה החישובית הוא מעבר streaming יחיד על `stop_times.txt` (15.7M שורות, 816 MB), אשר לעולם אינו נטען לזיכרון כטבלה.

**שאלת המחקר שאליה משיב שלב זה:** *מהו האובייקט הפורמלי הנכון לחקירה?* כל מה שנעשה בהמשך (centrality, קהילות, עמידות, סימולציות תקיפה) מוגדר מעל `G`, ולפיכך ההגדרה של `V`, `E` ו-`W` וההנחות העומדות בבסיסן חייבות להיאמר במפורש ולהיבדק, ולא להילקח כמובנות מאליהן.

## קלט

- `outputs/nb/01_data_preparation/tables/stops_clean.csv` - תכונות התחנות לאחר ניקוי (שם, lat/lon, מחוז, מטרופולין). **מיוצר על ידי הנוטבוק `01_data_preparation`.**
- `israel-public-transportation/stop_times.txt` - פיד זמני התחנות הגולמי של GTFS. בגודל 816 MB, **אינו מנוהל תחת git**; מורד לפי דרישה מ-Google Drive על ידי אחד התאים שבהמשך.

## פלט (הכול תחת `outputs/nb/02_graph_construction/`)

| נתיב | תוכן |
|---|---|
| `graph_directed.pkl` | `networkx.DiGraph` - גרף שכנויות-נסיעה, תכונת הקשת `weight` = מספר נסיעות למקטע |
| `graph_undirected.pkl` | `networkx.Graph` - ההיטל הלא-מכוון, `weight` = סכום שני הכיוונים |
| `tables/nodes.csv` | שורה אחת לכל צומת, עם תכונות התחנה שלו |
| `tables/edges.csv` | שורה אחת לכל מקטע **מכוון**, עם `trip_frequency` |
| `tables/top_segments.csv` | המקטעים העמוסים ביותר, לבדיקת שפיות מהירה |
| `tables/graph_build_summary.json` | ספירת צמתים/קשתות, צפיפות, סטטיסטיקות בנייה, מוני שלמות נתונים |
| `figures/top_segments.png`, `figures/weight_distribution.png` | איורים לבדיקת שפיות |

נוטבוקים מאוחרים יותר טוענים את `outputs/nb/02_graph_construction/graph_undirected.pkl` (ואת הגרף המכוון היכן שהכיוון משמעותי).

## אתחול סביבת העבודה

התא שלהלן מאפשר להריץ את הנוטבוק הן על עותק מקומי של המאגר והן על Google Colab. הוא מאתר את שורש המאגר (ומשכפל אותו אם אנו על Colab והוא אינו קיים), מעביר אליו את תיקיית העבודה, ויוצר את תיקיית `outputs/nb` המשותפת. הפונקציה `_ensure(...)` מתקינה אך ורק את החבילות שחסרות בפועל, כך שהרצה חוזרת של הנוטבוק אינה משלמת את מחיר הפנייה ל-pip. שום דבר כאן אינו נוגע בפלטי הדוח הקיימים תחת `outputs/tables`, `outputs/figures` או `outputs/rail`.

In [ ]:
# --- Environment bootstrap (safe to re-run, works locally and on Google Colab) ---
import os, sys, subprocess
from pathlib import Path

def _ensure(*pkgs):
    """Install only the packages that are actually missing."""
    import importlib.util
    alias = {"scikit-learn": "sklearn", "python-louvain": "community",
             "python-bidi": "bidi", "node2vec": "node2vec"}
    missing = [p for p in pkgs
               if importlib.util.find_spec(alias.get(p, p.replace("-", "_"))) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_repo_root():
    """Find the repo locally; on Colab, clone it."""
    here = Path(os.getcwd()).resolve()
    for cand in [here, *here.parents]:
        if (cand / "israel-public-transportation").is_dir():
            return cand
    target = Path("/content/israel-transit-network-resilience")
    if not target.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/seanfourman/israel-transit-network-resilience.git",
                        str(target)], check=True)
    return target

REPO = find_repo_root()
os.chdir(REPO)
DATA = REPO / "israel-public-transportation"
OUT = REPO / "outputs" / "nb"
OUT.mkdir(parents=True, exist_ok=True)
print("Repo root:", REPO)

## ספריות, תיקיות השלב ובקרות עלות

אנו נזקקים ל-`pandas` (טבלאות), ל-`networkx` (אובייקטי הגרף) ול-`matplotlib` (איורי בדיקת השפיות). המודול `csv` מספריית התקן הוא זה שקורא בפועל את הפיד בגודל 816 MB - הוא מחזיר שורה אחת בכל פעם, וזוהי כל מהותו של תכן ה-streaming.

שלושה קבועים שולטים בזמן הריצה ורוכזו כאן כדי שיהיה קל לשנותם:

- `SORT_CHECK_ROWS = 500_000` - כמה שורות דוגם מעבר האימות של הסדר. כשתי שניות; ניתן להגדיל את הערך אם רוצים רמת ביטחון גבוהה יותר.
- `PROGRESS_EVERY = 2_000_000` - כל כמה שורות מדפיס המעבר הראשי דיווח התקדמות. עניין קוסמטי בלבד.
- `FULL_INTEGRITY_CHECK = True` - סופר הפרות סדר גם על פני **כל** 15.7M השורות במהלך המעבר הראשי. העלות היא פענוח `int()` אחד לכל שורה (כמה שניות נוספות על מעבר שכבר נמשך 2-5 דקות). יש להגדיר `False` אם רוצים רק את הבדיקה המדגמית.

הערך `csv.field_size_limit` מוגדל משום שכמה שורות בפיד ארוכות באופן חריג, והמגבלה שכברירת מחדל הייתה מפילה את הקריאה.

In [ ]:
_ensure("pandas", "networkx", "matplotlib")

import csv, json, pickle, time
from collections import defaultdict

import pandas as pd
import networkx as nx

# A handful of rows in stop_times.txt are very long; raise the csv field limit up front.
csv.field_size_limit(10_000_000)

# ---- cost knobs (see markdown above) ----
SORT_CHECK_ROWS = 500_000        # rows sampled by the ordering-verification pass
PROGRESS_EVERY = 2_000_000       # progress print interval in the main streaming pass
FULL_INTEGRITY_CHECK = True      # track ordering violations over the whole file too

# ---- this stage's own output folder ----
STAGE = OUT / "02_graph_construction"
TABLES = STAGE / "tables"
FIGURES = STAGE / "figures"
TABLES.mkdir(parents=True, exist_ok=True)
FIGURES.mkdir(parents=True, exist_ok=True)

print("pandas", pd.__version__, "| networkx", nx.__version__)
print("Stage output folder:", STAGE)

## המודל הפורמלי של הגרף

הקורס מחייב הגדרה פורמלית מפורשת של האובייקט הנחקר, ולהלן היא.

יהי $T$ אוסף הנסיעות (trips) בפיד ה-GTFS. כל נסיעה $t \in T$ היא סדרה **סדורה** של ביקורים בתחנות

$$t = \left( s^{t}_{1},\, s^{t}_{2},\, \dots,\, s^{t}_{k_t} \right),$$

כאשר $s^{t}_{i}$ היא התחנה המשורתת במיקום $i$ של הנסיעה (מיקום = השדה `stop_sequence` של GTFS), ו-$k_t$ הוא מספר התחנות באותה נסיעה.

נגדיר את **גרף שכנויות-הנסיעה** הממושקל

$$G = (V,\, E,\, W)$$

- **צמתים.** $V = \{\, v : v \text{ is a stop incident to at least one segment} \,\}$. באופן קונקרטי, תחנה היא צומת אם קיימת נסיעה הנכנסת אליה או יוצאת ממנה. תחנה המופיעה בפיד אך לעולם אין לה קודמת ואין לה עוקבת (נסיעה מנוונת בעלת תחנה יחידה) *אינה* צומת - ראו את ההערה על תחנות מבודדות שלהלן.
- **קשתות.** $E \subseteq V \times V$, כאשר
  $$ (u,v) \in E \iff \exists\, t \in T,\ \exists\, i < k_t : \ s^{t}_{i} = u \ \wedge\ s^{t}_{i+1} = v \ \wedge\ u \neq v .$$
  במילים: קשת קיימת בדיוק כאשר נסיעה כלשהי עוברת מ-$u$ **ישירות** ל-$v$ ללא תחנת ביניים. זהו יחס של *שירות*, לא יחס גאוגרפי - שתי תחנות במרחק 50 מ' זו מזו שאין ביניהן קו מקשר **אינן** מחוברות.
- **משקלים.** $W : E \to \mathbb{N}$, כאשר
  $$ W(u,v) = \left| \{\, (t,i) \ : \ s^{t}_{i} = u,\ s^{t}_{i+1} = v \,\} \right| ,$$
  כלומר מספר הנסיעות המשתמשות במקטע $u \to v$. המשקל הוא אפוא מדד של **תדירות / קיבולת**: משקל גבוה משמעו שנסיעות שירות רבות עוברות מדי יום על אותו חיבור.

### ההיטל הלא-מכוון

מרבית הניתוח (קשירות, קהילות, פרקולציה תחת הסרת צמתים) אינו תלוי כיוון, ולכן אנו בונים גם את ההיטל הלא-מכוון $\tilde{G} = (V, \tilde{E}, \tilde{W})$ שבו

$$ \{u,v\} \in \tilde{E} \iff (u,v) \in E \ \vee\ (v,u) \in E, \qquad \tilde{W}(\{u,v\}) = W(u,v) + W(v,u) $$

(כאשר כיוון חסר תורם $0$). הבחירה לסכום, במקום למצע או לקחת מקסימום, היא מכוונת: כך המשקל הלא-מכוון שומר על משמעותו הפיזיקלית כ*מספר הכולל של נסיעות השירות החוצות את החיבור בשני הכיוונים*.

### דוגמה מפורטת

שלוש נסיעות מעל ארבע תחנות $A, B, C, D$:

| נסיעה | רצף התחנות |
|---|---|
| $t_1$ | $A \to B \to C$ |
| $t_2$ | $A \to B \to D$ |
| $t_3$ | $C \to B \to A$ |

הגרף המכוון: $V = \{A,B,C,D\}$ וכן

$$E = \{(A,B), (B,C), (B,D), (C,B), (B,A)\}, \quad W(A,B) = 2,\ W(B,C) = W(B,D) = W(C,B) = W(B,A) = 1 .$$

ההיטל הלא-מכוון: $\tilde{E} = \{\{A,B\}, \{B,C\}, \{B,D\}\}$ כאשר

$$\tilde{W}(\{A,B\}) = W(A,B) + W(B,A) = 2 + 1 = 3, \quad \tilde{W}(\{B,C\}) = 1 + 1 = 2, \quad \tilde{W}(\{B,D\}) = 1 + 0 = 1 .$$

שימו לב ש-$B$ כבר בולט עם דרגה 3 בעוד ש-$A$, $C$, $D$ הם בעלי דרגה 1 - זהו בדיוק מבנה ה-hub שהנוטבוקים המאוחרים יותר העוסקים ב-centrality מכמתים בקנה מידה ארצי.

### החלטות מידול, מוצהרות בגלוי

1. **לולאות עצמיות מושמטות.** אם נסיעה מונה את אותה תחנה פעמיים ברציפות (הדבר קורה בפיד, בדרך כלל כתוצר לוואי של תזמון), אנו מדלגים עליה במקום ליצור לולאה $u \to u$, שאינה נושאת מידע קשירות כלשהו. מספר הדילוגים מדווח.
2. **המשקלים סופרים נסיעות, לא נוסעים.** ב-GTFS אין נתוני ביקוש נוסעים. התדירות היא ה-proxy הטוב ביותר הזמין למידת התלות של השירות בחיבור מסוים.
3. **הגרף הוא תצלום סטטי יחיד** של כל תקופת הפיד - ללא פילוח לפי שעות היום או לפי ימי חול מול סוף שבוע.
4. **תחנות מבודדות אינן נכללות ב-$V$.** תחנה ללא מקטע נכנס וללא מקטע יוצא הייתה צומת מבודד ומעוותת את ערכי הצפיפות והדרגה הממוצעת; תחנות כאלה נספרות ומדווחות בנפרד בסיכום.

### שתי הפונקציות המרכזיות, בהדגמה על הדוגמה המפורטת

לפני שאנו מריצים דבר על 15.7M שורות, אנו מגדירים את שתי הפונקציות המממשות את ההגדרה שלעיל ובודקים אותן מול דוגמת הצעצוע, שאת תוצאתה כבר חישבנו ידנית. `edges_from_trips` הופכת רצפי תחנות לפונקציית המשקל $W$, ו-`undirected_from_directed` מבצעת את היטל הסכימה. אותה `undirected_from_directed` עצמה נעשה בה שימוש חוזר על הגרף האמיתי בהמשך, ולכן אימותה כאן מאמת אותה גם שם.

In [ ]:
def edges_from_trips(trips):
    """Count directed segments u->v over an iterable of stop sequences.

    Implements W(u,v) = #{(t,i) : s_i^t = u and s_{i+1}^t = v}, skipping self-loops.
    """
    counts = defaultdict(int)
    for seq in trips:
        for u, v in zip(seq, seq[1:]):
            if u != v:
                counts[(u, v)] += 1
    return counts


def undirected_from_directed(D):
    """Undirected projection: weights of the two directions are SUMMED."""
    G = nx.Graph()
    for u, v, data in D.edges(data=True):
        w = data["weight"]
        if G.has_edge(u, v):
            G[u][v]["weight"] += w
        else:
            G.add_edge(u, v, weight=w)
    return G


# --- the worked example from the markdown above ---
demo_trips = [["A", "B", "C"], ["A", "B", "D"], ["C", "B", "A"]]
demo_counts = edges_from_trips(demo_trips)

demo_D = nx.DiGraph()
for (u, v), c in demo_counts.items():
    demo_D.add_edge(u, v, weight=c)
demo_G = undirected_from_directed(demo_D)

print("Directed edges  E, W:")
for u, v, d in sorted(demo_D.edges(data=True)):
    print(f"  {u} -> {v} : W = {d['weight']}")
print("\nUndirected edges  E~, W~:")
for u, v, d in sorted(demo_G.edges(data=True)):
    print(f"  {{{u},{v}}} : W~ = {d['weight']}")
print("\nDegrees in the projection:", dict(demo_G.degree()))

# Assert the hand-computed answer, so a silent regression here fails loudly.
assert demo_D.number_of_nodes() == 4 and demo_D.number_of_edges() == 5
assert demo_D["A"]["B"]["weight"] == 2
assert demo_G.number_of_edges() == 3
assert demo_G["A"]["B"]["weight"] == 3 and demo_G["B"]["C"]["weight"] == 2
print("\nWorked example matches the definition.")

## תלות בנתונים חיצוניים: `stop_times.txt`

הקובץ `stop_times.txt` הוא בגודל 816 MB - הרבה מעל מגבלת גודל הקובץ של GitHub - ולכן הוא **אינו** נמצא במאגר. התא שלהלן מוריד אותו מ-Google Drive בהרצה הראשונה, ומדלג על ההורדה אם הקובץ כבר קיים. זוהי תלות הרשת החיצונית היחידה של הנוטבוק; כל היתר נמצא במאגר או מיוצר על ידי נוטבוק 01.

ההורדה נמשכת כמה דקות בהרצה ראשונה על Colab.

In [ ]:
# stop_times.txt is 816MB and is not tracked in git - fetch it on demand.
_ensure("gdown")
import gdown
STOP_TIMES = DATA / "stop_times.txt"
if not STOP_TIMES.exists():
    gdown.download(id="1V_yPAWXV6mGTFGrfiosah5LngcLZnviW",
                   output=str(STOP_TIMES), quiet=False)
print("stop_times.txt:", round(STOP_TIMES.stat().st_size / 1024**2, 1), "MB")

## תכונות התחנות משלב 01

קבוצת הקשתות מגיעה מ-`stop_times.txt`, אך על הצמתים לשאת מידע בעל משמעות: שם בעברית, קואורדינטות, ותוויות המחוז / אזור המטרופולין שהוקצו במהלך הניקוי. אלה מגיעים מ-`stops_clean.csv`, שהוא הפלט של הנוטבוק **01_data_preparation**.

אנו מחפשים את הקובץ בתיקיית הפלט של שלב 01 ונכשלים בהודעה מפורשת ומעשית אם אינו נמצא - גרף חסר תכונות באופן שקט היה שובר את הניתוח הגאוגרפי כמה נוטבוקים מאוחר יותר, וזוהי תקלה יקרה בהרבה. תחנות המופיעות ב-`stop_times.txt` אך אינן מופיעות בטבלת התחנות המנוקה (למשל כאלה שהוסרו בשל קואורדינטות לא תקינות) הופכות בכל זאת לצמתים, אלא שהן פשוט מקבלות תכונות ריקות; מספר התחנות מסוג זה מדווח.

In [ ]:
STAGE01 = OUT / "01_data_preparation"
_candidates = [STAGE01 / "tables" / "stops_clean.csv", STAGE01 / "stops_clean.csv"]
STOPS_CLEAN = next((p for p in _candidates if p.exists()), None)
if STOPS_CLEAN is None:
    raise FileNotFoundError(
        "stops_clean.csv not found - run notebook 01_data_preparation first.\n"
        "Looked in:\n  " + "\n  ".join(str(p) for p in _candidates)
    )


def _to_float(x):
    """Coordinates arrive as strings and may be blank; return None instead of raising."""
    try:
        return float(x)
    except (TypeError, ValueError):
        return None


stops_df = pd.read_csv(STOPS_CLEAN, dtype=str, keep_default_na=False, encoding="utf-8-sig")
ATTR = {
    r["stop_id"]: {
        "stop_name": r.get("stop_name", "") or "",
        "lat": _to_float(r.get("stop_lat")),
        "lon": _to_float(r.get("stop_lon")),
        "region": r.get("region", "") or "",
        "metro": r.get("metro", "") or "",
    }
    for r in stops_df.to_dict("records")
}
DEFAULT_ATTR = {"stop_name": "", "lat": None, "lon": None, "region": "", "metro": ""}

print(f"Loaded attributes for {len(ATTR):,} stops from {STOPS_CLEAN}")
print("Columns available:", list(stops_df.columns))

## הנחת הסדר - ובדיקה מפורשת שלה

בניית ה-streaming נשענת על הנחה אחת שראוי לנסחה במפורש:

> **הנחה (סדר הפיד).** הקובץ `stop_times.txt` ממוין לפי `trip_id`, ובתוך כל `trip_id` לפי `stop_sequence` בסדר עולה. כתוצאה מכך, כל שורות נסיעה מסוימת מהוות בלוק רציף אחד, ושתי שורות עוקבות של אותה נסיעה מתארות תחנות עוקבות שלה.

זוהי ההמלצה של תקן GTFS וכך אכן מתנהג הפיד הישראלי בפועל, וזו הסיבה שביכולתנו לבנות את הגרף בזיכרון $O(1)$ לכל שורה במקום לקבץ 15.7M שורות בזיכרון. אולם זוהי הנחה לגבי ייצוא נתונים של גורם אחר, ואם היא הייתה מופרת, הקשתות המתקבלות היו **שגויות בשקט** - היינו מחברים תחנות שאינן עוקבות בפועל. זהו סוג הכשל הגרוע ביותר: אין שגיאה, פשוט גרף שגוי בעדינות המזין את כל התוצאות בהמשך.

לכן אנו בודקים אותה. המעבר שלהלן קורא את `SORT_CHECK_ROWS` השורות הראשונות ומחפש שני סוגי הפרה:

1. **נסיגות ב-`stop_sequence`** - שורה שערך ה-`stop_sequence` שלה אינו גדול ממש מזה של השורה הקודמת של אותה נסיעה (שורות בתוך נסיעה שאינן בסדר הנכון).
2. **בלוקי נסיעה משולבים זה בזה** - `trip_id` המופיע שוב לאחר שכבר עברנו לנסיעה אחרת (שורות של נסיעה שאינן רציפות).

אם אחד המונים אינו אפס, התא מדפיס אזהרה בולטת המודיעה שאין לסמוך על תוצאת ה-streaming ושיש למיין תחילה את הקובץ (`sort -t, -k1,1 -k5,5n`, או בנייה מבוססת group-by ב-pandas). גרסת הקובץ המלא של אותם שני מונים רצה גם היא במהלך המעבר הראשי כאשר `FULL_INTEGRITY_CHECK` פעיל, כך שהמדגם משמש כאזהרה מוקדמת מהירה ולא כקו ההגנה היחיד.

In [ ]:
def verify_sort_assumption(path, max_rows=SORT_CHECK_ROWS, report_examples=5):
    """Sample the head of stop_times.txt and check it is sorted by (trip_id, stop_sequence).

    Returns a dict of counters; prints a loud warning if the assumption is violated.
    """
    seq_regressions, block_revisits = [], []
    rows = 0
    trip_blocks = 0
    seen_trips = set()

    with open(path, encoding="utf-8-sig", newline="") as f:
        reader = csv.reader(f)
        header = next(reader)
        ti = header.index("trip_id")
        qi = header.index("stop_sequence")

        prev_trip, prev_seq = None, None
        for row in reader:
            rows += 1
            trip = row[ti]
            try:
                seq = int(row[qi])
            except (ValueError, IndexError):
                seq = None

            if trip != prev_trip:
                trip_blocks += 1
                if trip in seen_trips:
                    block_revisits.append((rows, trip))
                seen_trips.add(trip)
                prev_seq = None
            elif seq is not None and prev_seq is not None and seq <= prev_seq:
                seq_regressions.append((rows, trip, prev_seq, seq))

            prev_trip, prev_seq = trip, seq
            if rows >= max_rows:
                break

    result = {
        "rows_sampled": rows,
        "trip_blocks_sampled": trip_blocks,
        "distinct_trips_sampled": len(seen_trips),
        "stop_sequence_regressions": len(seq_regressions),
        "interleaved_trip_blocks": len(block_revisits),
        "assumption_holds": not seq_regressions and not block_revisits,
    }

    print(f"Sort-assumption check over the first {rows:,} rows "
          f"({len(seen_trips):,} distinct trips):")
    print(f"  stop_sequence regressions : {len(seq_regressions):,}")
    print(f"  interleaved trip blocks   : {len(block_revisits):,}")

    if result["assumption_holds"]:
        print("  OK - the sample is sorted by (trip_id, stop_sequence).")
    else:
        bar = "!" * 78
        print("\n" + bar)
        print("WARNING: THE FEED IS NOT SORTED BY (trip_id, stop_sequence).")
        print("The streaming edge construction below assumes consecutive rows of the same")
        print("trip are consecutive stops. That assumption is FALSE for this file, so the")
        print("edges it produces WILL BE WRONG. Sort the file first, e.g.")
        print("    sort -t, -k1,1 -k5,5n stop_times.txt > stop_times_sorted.txt")
        print("(keeping the header) or rebuild the edges with a group-by over trip_id.")
        for ex in seq_regressions[:report_examples]:
            print(f"  example regression: row {ex[0]:,} trip {ex[1]} seq {ex[2]} -> {ex[3]}")
        for ex in block_revisits[:report_examples]:
            print(f"  example interleaved trip: row {ex[0]:,} trip {ex[1]} reappears")
        print(bar + "\n")

    return result


sort_check = verify_sort_assumption(STOP_TIMES)

## מעבר ה-streaming על 15.7M השורות

זהו השלב היקר: כ**2-5 דקות**, והסיבה היחידה לכך שהנוטבוק אינו מיידי. הקובץ נקרא באמצעות `csv.reader`, שורה אחת בכל פעם, ואנו שומרים אך ורק את:

- `edge_count` - `dict` הממפה `(u, v)` למספר הנסיעות המשתמשות באותו מקטע (כ-52k רשומות, זניח);
- `active_stops` / `trips_seen` - קבוצות המשמשות לדיווח;
- `stops_per_trip` - התפלגות אורכי הנסיעות;
- ערכי `trip_id`, `stop_id` ו-`stop_sequence` של השורה הקודמת.

כלל הקשתות הוא מימוש ישיר של ההגדרה הפורמלית: אם השורה הנוכחית שייכת לאותה נסיעה כמו השורה הקודמת, ושתי התחנות שונות זו מזו, מגדילים את `W(prev_stop, stop)` באחד. שינוי ב-`trip_id` מאפס את המצב, כך שלעולם אין נוצרת קשת החוצה גבול בין נסיעות.

מכיוון שממילא אנו נוגעים בכל שורה, אותם שני מוני שלמות של הבדיקה המדגמית מתוחזקים על פני הקובץ כולו (כאשר `FULL_INTEGRITY_CHECK` פעיל) - הדבר כמעט חינמי והוא משדרג את הבדיקה המדגמית לבדיקה מלאה. שני המונים מגיעים לסיכום הנשמר, כך שההנחה מתועדת עם ראיות ולא רק נטענת.

In [ ]:
def stream_trip_edges(path, progress_every=PROGRESS_EVERY, integrity=FULL_INTEGRITY_CHECK):
    """Single streaming pass over stop_times.txt building the segment weight function W.

    Assumes the file is sorted by (trip_id, stop_sequence) - verified by the cell above
    and re-verified here over the full file when `integrity` is True.
    Memory is O(|E| + |T|), never O(rows).
    """
    edge_count = defaultdict(int)
    active_stops = set()
    trips_seen = set()
    stops_per_trip = defaultdict(int)
    stop_use_count = defaultdict(int)   # scheduled stop calls per stop
    rows_read = 0
    self_loops_skipped = 0
    seq_regressions = 0
    interleaved_trip_blocks = 0
    t0 = time.time()

    with open(path, encoding="utf-8-sig", newline="") as f:
        reader = csv.reader(f)
        header = next(reader)
        ti = header.index("trip_id")
        si = header.index("stop_id")
        qi = header.index("stop_sequence") if "stop_sequence" in header else None

        prev_trip, prev_stop, prev_seq = None, None, None
        for row in reader:
            rows_read += 1
            trip = row[ti]
            stop = row[si]
            active_stops.add(stop)
            stop_use_count[stop] += 1
            stops_per_trip[trip] += 1

            if trip != prev_trip:
                # new trip block starts here
                if integrity and trip in trips_seen:
                    interleaved_trip_blocks += 1
                trips_seen.add(trip)
                prev_seq = None
            else:
                # same trip as the previous row -> this pair is a segment
                if prev_stop is not None:
                    if prev_stop != stop:
                        edge_count[(prev_stop, stop)] += 1
                    else:
                        self_loops_skipped += 1

            if integrity and qi is not None:
                try:
                    seq = int(row[qi])
                except (ValueError, IndexError):
                    seq = None
                if seq is not None and prev_seq is not None and seq <= prev_seq:
                    seq_regressions += 1
                prev_seq = seq

            prev_trip, prev_stop = trip, stop

            if progress_every and rows_read % progress_every == 0:
                print(f"    {rows_read:,} rows | {len(edge_count):,} unique segments "
                      f"| {time.time() - t0:,.0f}s")

    spt = list(stops_per_trip.values())
    build_stats = {
        "stop_times_rows": rows_read,
        "active_stops": len(active_stops),
        "active_trips": len(trips_seen),
        "directed_edges": len(edge_count),
        "min_stops_per_trip": int(min(spt)) if spt else 0,
        "mean_stops_per_trip": round(sum(spt) / len(spt), 2) if spt else 0,
        "max_stops_per_trip": int(max(spt)) if spt else 0,
        "self_loops_skipped": self_loops_skipped,
        "full_file_integrity_check": bool(integrity),
        "full_file_stop_sequence_regressions": seq_regressions if integrity else None,
        "full_file_interleaved_trip_blocks": interleaved_trip_blocks if integrity else None,
        "elapsed_seconds": round(time.time() - t0, 1),
    }
    return edge_count, active_stops, stop_use_count, build_stats


print(f"Streaming {STOP_TIMES.name} (~15.7M rows, this takes a few minutes) ...")
edge_count, active_stops, stop_use_count, build_stats = stream_trip_edges(STOP_TIMES)

print("\nBuild statistics:")
for k, v in build_stats.items():
    print(f"  {k}: {v}")

if build_stats["full_file_integrity_check"] and (
    build_stats["full_file_stop_sequence_regressions"]
    or build_stats["full_file_interleaved_trip_blocks"]
):
    print("\n" + "!" * 78)
    print("WARNING: ordering violations found over the FULL file - the edges above are")
    print("not trustworthy. See the sort-assumption section for how to fix the feed.")
    print("!" * 78)

## מימוש שני אובייקטי הגרף

כעת המקטעים שנספרו הופכים לאובייקטים ממשיים של `networkx`. הגרף המכוון הוא תמונה חד-חד-ערכית של `edge_count`; הגרף הלא-מכוון מיוצר על ידי `undirected_from_directed` שכבר נבדקה, כלומר על ידי סכימת שני הכיוונים. תכונות הצמתים משלב 01 מוצמדות לשניהם.

שימו לב להשלכה של הגדרת $V$: הצמתים נגזרים מן ה*קשתות*, ולכן תחנה שהופיעה ב-`stop_times.txt` אך מעולם לא היה לה שכן אינה נמצאת בגרף. אנו מחשבים את הפער הזה במפורש במקום לתת לו לחלוף ללא תשומת לב - מדובר בקומץ תחנות מתוך כ-30k, ואם הוא יגדל אי פעם, הדבר מסמן בעיה בנתונים.

In [ ]:
def build_graphs(edge_count, attr):
    """Build the directed trip-adjacency graph and its summed undirected projection."""
    D = nx.DiGraph()
    for (u, v), c in edge_count.items():
        D.add_edge(u, v, weight=c)
    for n in D.nodes():
        D.nodes[n].update(attr.get(n, DEFAULT_ATTR))

    G = undirected_from_directed(D)
    for n in G.nodes():
        G.nodes[n].update(attr.get(n, DEFAULT_ATTR))
    return G, D


G, D = build_graphs(edge_count, ATTR)

isolated_active_stops = sorted(active_stops - set(G.nodes()))
nodes_without_attributes = [n for n in G.nodes() if n not in ATTR]

print(f"Directed graph   : {D.number_of_nodes():,} nodes, {D.number_of_edges():,} edges")
print(f"Undirected graph : {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")
print(f"Density (undirected): {nx.density(G):.6f}")
print(f"Average degree      : {sum(d for _, d in G.degree()) / G.number_of_nodes():.2f}")
print(f"\nStops served by a trip but with no segment (excluded from V): "
      f"{len(isolated_active_stops)}  {isolated_active_stops[:10]}")
print(f"Nodes with no attributes in stops_clean.csv: {len(nodes_without_attributes)}")
print(f"Connected components (undirected): {nx.number_connected_components(G):,}")

## שמירת תוצרי השלב

כל השלבים הבאים קוראים את הקבצים הללו במקום לבצע streaming חוזר על הפיד בגודל 816 MB, ולכן תא זה הוא מה שהופך את שאר הפרויקט לזול להרצה.

- שני קובצי ה-pickle הם הגרפים עצמם, הנכתבים לשורש תיקיית השלב;
- `tables/nodes.csv` ו-`tables/edges.csv` הם הצורה הקריאה לאדם / הניידת (UTF-8 עם BOM, כדי ששמות בעברית ייפתחו כראוי ב-Excel);
- `tables/graph_build_summary.json` מתעד את המספרים המרכזיים ואת כל מוני השלמות, וזה מה שהדוח מצטט.

דבר אינו נכתב אל `outputs/tables`, `outputs/figures` או `outputs/rail`.

In [ ]:
with open(STAGE / "graph_undirected.pkl", "wb") as f:
    pickle.dump(G, f)
with open(STAGE / "graph_directed.pkl", "wb") as f:
    pickle.dump(D, f)

nodes_df = pd.DataFrame(
    [{"stop_id": nid, **data} for nid, data in G.nodes(data=True)]
)
# Scheduled stop calls: how many times each stop appears in stop_times.txt.
# This is the service-volume measure the equity analysis (notebook 08) needs.
# It can only be counted during the streaming pass - it is not recoverable
# from the graph, because edge weights merge the two travel directions.
nodes_df["stop_use_count"] = (
    nodes_df["stop_id"].map(stop_use_count).fillna(0).astype(int)
)
nodes_df.to_csv(TABLES / "nodes.csv", index=False, encoding="utf-8-sig")

edges_df = pd.DataFrame(
    [{"from_stop": u, "to_stop": v, "trip_frequency": data["weight"]}
     for u, v, data in D.edges(data=True)]
)
edges_df.to_csv(TABLES / "edges.csv", index=False, encoding="utf-8-sig")

summary = {
    "graph_type": "trip_adjacency",
    "num_nodes": G.number_of_nodes(),
    "num_edges_undirected": G.number_of_edges(),
    "num_edges_directed": D.number_of_edges(),
    "avg_degree": round(sum(d for _, d in G.degree()) / G.number_of_nodes(), 2),
    "density": round(nx.density(G), 6),
    "connected_components": nx.number_connected_components(G),
    "largest_component_size": len(max(nx.connected_components(G), key=len)),
    "active_stops_without_segments": len(isolated_active_stops),
    "nodes_without_attributes": len(nodes_without_attributes),
    "sort_assumption_sample_check": sort_check,
    "build_stats": build_stats,
}
with open(TABLES / "graph_build_summary.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)

print("Saved:")
for p in [STAGE / "graph_undirected.pkl", STAGE / "graph_directed.pkl",
          TABLES / "nodes.csv", TABLES / "edges.csv",
          TABLES / "graph_build_summary.json"]:
    print(f"  {p}  ({p.stat().st_size / 1024:,.0f} KB)")

print("\nSummary:")
print(json.dumps({k: v for k, v in summary.items()
                  if k not in ("build_stats", "sort_assumption_sample_check")},
                 indent=2, ensure_ascii=False))

## תמיכה בתוויות בעברית באיורים

שמות התחנות הם בעברית. Matplotlib משרטטת גליפים משמאל לימין ואינה מיישמת את אלגוריתם הדו-כיווניות (bidi) של Unicode, ולכן תוויות בעברית יוצאות הפוכות. התיקון שלהלן מריץ את אלגוריתם ה-bidi על כל אובייקט טקסט פעם אחת, לפני ששורטט איור כלשהו.

In [ ]:
# Stop names are Hebrew. Matplotlib does not apply the Unicode bidi algorithm, so
# Hebrew labels render reversed. Patch it once, before drawing any figure.
_ensure("python-bidi")
import re
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.text as mtext
from bidi.algorithm import get_display

_HEBREW_RE = re.compile(r"[\u0590-\u05FF]")

def fix_he(text):
    """Return display-ordered text. Non-Hebrew is returned untouched."""
    if not isinstance(text, str) or not _HEBREW_RE.search(text):
        return text
    return get_display(text)

def install_hebrew():
    # Arial exists on Windows; DejaVu Sans ships with matplotlib and covers Hebrew.
    matplotlib.rcParams["font.family"] = ["Arial", "DejaVu Sans"]
    matplotlib.rcParams["axes.unicode_minus"] = False
    if getattr(mtext.Text, "_bidi_patched", False):
        return
    _orig = mtext.Text.set_text
    def set_text(self, s):
        if isinstance(s, str) and getattr(self, "_bidi_display", None) == s:
            return _orig(self, s)
        fixed = fix_he(s)
        if isinstance(fixed, str):
            self._bidi_display = fixed
        return _orig(self, fixed)
    mtext.Text.set_text = set_text
    mtext.Text._bidi_patched = True

install_hebrew()

## בדיקת שפיות: האם הגרף נראה כמו רשת תחבורה ציבורית אמיתית?

בנייה יכולה להסתיים ללא שגיאה ועדיין להיות שגויה, ולכן אנו בוחנים את התוצאה לפני שאנו נותנים בה אמון. שתי בדיקות:

1. **המקטעים העמוסים ביותר.** אם החיבורים בעלי המשקל הגבוה ביותר הם צירי תדירות גבוהה מוכרים בין תחנות אמיתיות ומזוהות בשמן, כלל הקשתות עושה את מה שאנו סבורים שהוא עושה. אילו היו אלה זוגות פרווריים אקראיים, הנחת הסדר הייתה נעשית חשודה.
2. **התפלגות המשקלים.** תדירות תחבורה ציבורית היא בעלת זנב כבד מובהק - רוב המקטעים משורתים על ידי קומץ נסיעות ביום ומעטים נושאים מאות. היסטוגרמה בסקאלה לוגריתמית אמורה להראות זאת, והתפלגות שהייתה נראית אחידה הייתה מעידה שמשהו השתבש בספירה.

טבלת המקטעים המובילים נשמרת גם היא אל `tables/top_segments.csv`.

In [ ]:
TOP_N = 15

name_of = {n: (G.nodes[n].get("stop_name") or n) for n in G.nodes()}
top = sorted(G.edges(data=True), key=lambda e: e[2]["weight"], reverse=True)[:TOP_N]
top_df = pd.DataFrame([
    {"stop_a": u, "stop_b": v,
     "name_a": name_of[u], "name_b": name_of[v],
     "trips_both_directions": d["weight"]}
    for u, v, d in top
])
top_df.to_csv(TABLES / "top_segments.csv", index=False, encoding="utf-8-sig")
display(top_df)

fig, ax = plt.subplots(figsize=(10, 6))
labels = [f"{r.name_a} - {r.name_b}" for r in top_df.itertuples()][::-1]
ax.barh(range(len(top_df)), top_df["trips_both_directions"][::-1], color="#3b6ea5")
ax.set_yticks(range(len(top_df)))
ax.set_yticklabels(labels, fontsize=8)
ax.set_xlabel("Trips per day (both directions)")
ax.set_title(f"Top {TOP_N} segments by service frequency")
fig.tight_layout()
fig.savefig(FIGURES / "top_segments.png", dpi=150)
plt.show()

weights = [d["weight"] for _, _, d in G.edges(data=True)]
fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(weights, bins=60, log=True, color="#3b6ea5", edgecolor="white", linewidth=0.4)
ax.set_xlabel("Segment weight (trips, both directions)")
ax.set_ylabel("Number of segments (log scale)")
ax.set_title("Edge-weight distribution of the trip-adjacency graph")
fig.tight_layout()
fig.savefig(FIGURES / "weight_distribution.png", dpi=150)
plt.show()

ws = pd.Series(weights)
print("Edge weight percentiles:")
print(ws.describe(percentiles=[0.5, 0.9, 0.99]).round(2).to_string())

## מסקנות

- פיד ה-GTFS הישראלי מייצר גרף שכנויות-נסיעה של כ**30.5k צמתים וכ-52k מקטעים מכוונים** (כ-51.8k לא-מכוונים), הנבנה מכ-15.7M שורות stop-time על פני כ-420k נסיעות, עם דרגה לא-מכוונת ממוצעת סביב **3.4** וצפיפות של כ-**1.1e-4**. זוהי רשת דלילה מאוד, כמעט מישורית, בעלת צורה של צירים - וזו בדיוק הסיבה לכך שהסרת מספר קטן של צמתים ממוקמים היטב עלולה לפגוע בה, וזו השאלה שבה מטפלים הנוטבוקים הבאים.
- **הנחת הסדר היא ממשית וכעת נבדקה, ולא הונחה.** הפיד ממוין לפי `(trip_id, stop_sequence)`, וגם הבדיקה המדגמית וגם מוני הקובץ המלא מדווחים על אפס הפרות בנתונים אלה. לכך יש חשיבות: אילו ההנחה הייתה מופרת, הנוטבוק היה מייצר גרף שנראה סביר אך שגוי, ואף שלב בהמשך לא היה מבחין בכך. המונים נשמרים ב-`graph_build_summary.json` כראיה.
- **מספר תחנות מושמטות באופן לגיטימי.** כ-3 תחנות מופיעות ב-`stop_times.txt` ללא קודמת וללא עוקבת כלשהי ולפיכך אינן צמתים, ולכן `num_nodes` נמוך במעט מ-`active_stops`. זוהי השלכה הגדרתית של $V$ ולא באג, אך הדבר מדווח ולא מוסתר.
- **המשקלים הם תדירות, לא ביקוש.** GTFS אינו נושא נתוני נוסעים, ולכן כל תוצאת "חשיבות" בפרויקט זה היא חשיבות *ברשת ההיצע*. מקטע עם 400 נסיעות יומיות משורת באופן אינטנסיבי; האם הוא גם בשימוש אינטנסיבי היא שאלה שנתונים אלה אינם יכולים לענות עליה, ומגבלה זו מתפשטת אל כל מסקנה בהמשך.
- התפלגות משקלי הקשתות היא בעלת זנב כבד מובהק: המקטע החציוני משורת על ידי מספר נסיעות בודד בלבד, בעוד שהאחוזון העליון נושא סדרי גודל יותר. החיבורים העמוסים ביותר מתרכזים בצירי מטרופולין תל אביב, וזהו הרמז הראשון לריכוזיות הגאוגרפית שהשלבים המאוחרים יותר מכמתים.